In [1]:
import sys
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.abspath('../src'))
from DataLoader import DataLoader
from DataSplitter import DataSplitter
from Transformer import Transformer
from PreProcessorSimple import PreProcessorSimple
from PreProcessorNN import PreProcessorNN
from PipelineBuilder import PipelineBuilder
from CrossValidation import CrossValidation

In [2]:
data_loader = DataLoader("../data/train.csv", "../data/test.csv")
train, test = data_loader.load()

In [3]:
train.shape, test.shape

((42000, 785), (28000, 784))

In [4]:
train.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
train.describe()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
count,42000.000000,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,...,42000.000000,42000.000000,42000.000000,42000.00000,42000.000000,42000.000000,42000.0,42000.0,42000.0,42000.0
mean,4.456643,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.219286,0.117095,0.059024,0.02019,0.017238,0.002857,0.0,0.0,0.0,0.0
std,2.887730,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.312890,4.633819,3.274488,1.75987,1.894498,0.414264,0.0,0.0,0.0,0.0
min,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
25%,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
50%,4.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
75%,7.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
max,9.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,254.000000,254.000000,253.000000,253.00000,254.000000,62.000000,0.0,0.0,0.0,0.0


In [6]:
data_splitter = DataSplitter(target_column="label")
X_train, X_test, y_train, y_test = data_splitter.split(train)
X, y = train.drop(columns=['label']), train['label']
X_train.shape, X_test.shape, y_train.shape, y_test.shape, X_train.shape[0]/(X_train.shape[0] + X_test.shape[0])

((33600, 784), (8400, 784), (33600,), (8400,), 0.8)

In [7]:
## ML pipeline builder
pipeline_builder = PipelineBuilder(train)

## KNN Classifier

In [8]:
pipeline = pipeline_builder.build(model_name="KNN")
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('transformer', ...), ('preprocessor', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers conta

In [9]:
## Hold-Out Validation score (one fold only)
cv = CrossValidation(pipeline=pipeline)
cv_score = cv.evaluate_one_fold(X_train, y_train, X_test, y_test)
print(cv_score)

Fit time: 3.860334634780884
Prediction time: 6.827239513397217

0.9698809523809524


In [10]:
## Hyper-parameter tunning
param_grid = {
    "model__n_neighbors": [1, 2, 3, 4, 5, 8, 10, 13, 15, 20, 30],
    "model__weights": ['distance'],
}
cv_knn_results = cv.hyper_param_tune_one_fold(X_train, y_train, X_test, y_test, param_grid, reduced=True, train_size=5000)
cv_knn_results

Params:  {'model__n_neighbors': 1, 'model__weights': 'distance'}
Fit time: 0.40400052070617676
Prediction time: 1.1326820850372314

Params:  {'model__n_neighbors': 2, 'model__weights': 'distance'}
Fit time: 0.5876317024230957
Prediction time: 0.981942892074585

Params:  {'model__n_neighbors': 3, 'model__weights': 'distance'}
Fit time: 0.3922545909881592
Prediction time: 0.8040826320648193

Params:  {'model__n_neighbors': 4, 'model__weights': 'distance'}
Fit time: 0.7063703536987305
Prediction time: 0.8457269668579102

Params:  {'model__n_neighbors': 5, 'model__weights': 'distance'}
Fit time: 0.5426616668701172
Prediction time: 1.1908588409423828

Params:  {'model__n_neighbors': 8, 'model__weights': 'distance'}
Fit time: 0.587580680847168
Prediction time: 1.0270533561706543

Params:  {'model__n_neighbors': 10, 'model__weights': 'distance'}
Fit time: 0.4986145496368408
Prediction time: 0.9834997653961182

Params:  {'model__n_neighbors': 13, 'model__weights': 'distance'}
Fit time: 0.47697

,model__n_neighbors,model__weights,fit_time,pred_time,epochs,final_loss,final_validation_score,score
3,4,distance,0.706370,0.845727,NaN,NaN,NaN,0.936667
2,3,distance,0.392255,0.804083,NaN,NaN,NaN,0.935357
4,5,distance,0.542662,1.190859,NaN,NaN,NaN,0.934643
5,8,distance,0.587581,1.027053,NaN,NaN,NaN,0.933452
0,1,distance,0.404001,1.132682,NaN,NaN,NaN,0.931429
1,2,distance,0.587632,0.981943,NaN,NaN,NaN,0.931429
6,10,distance,0.498615,0.983500,NaN,NaN,NaN,0.931071
7,13,distance,0.476980,1.021307,NaN,NaN,NaN,0.927619
8,15,distance,0.461547,0.822155,NaN,NaN,NaN,0.924048
9,20,distance,0.498733,1.010720,NaN,NaN,NaN,0.917857


In [11]:
## Create submission using best hyper-parameters
preds = cv.pipeline.fit(X, y).predict(test)
submission = pd.DataFrame({"ImageId": test.index+1, "Label": preds})
submission.to_csv("../data/submission_knn.csv", index=False)

## XGBoost Classifier

In [12]:
pipeline = pipeline_builder.build(model_name="XGBoost")
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('transformer', ...), ('preprocessor', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers conta

In [13]:
## Hold-Out Validation score (one fold only)
cv = CrossValidation(pipeline=pipeline)
cv_score = cv.evaluate_one_fold(X_train, y_train, X_test, y_test, reduced=True)
print(cv_score)

Fit time: 40.38600182533264
Prediction time: 0.7484912872314453

0.9239285714285714


In [14]:
## Hyper-parameter tunning
param_grid = {
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__max_depth": [3, 6, 10],
    "model__gamma": [0.01, 0.5, 1.0, 2.0],
}
#cv_knn_results = cv.hyper_param_tune_one_fold(X_train, y_train, X_test, y_test, param_grid, reduced=True, train_size=10000)
#cv_knn_results

In [15]:
## Create submission using best hyper-parameters
#preds = cv.pipeline.fit(X, y).predict(test)
#submission = pd.DataFrame({"ImageId": test.index+1, "Label": preds})
#submission.to_csv("../data/submission_xgb.csv", index=False)

# Neural Networks
### MLP (Multi-Layer Perceptron) - Sklearn

In [16]:
pipeline = pipeline_builder.build(model_name="MLP")
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('transformer', ...), ('preprocessor', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers conta

In [17]:
## Hold-Out Validation score (one fold only)
cv = CrossValidation(pipeline=pipeline)
cv_score = cv.evaluate_one_fold(X_train, y_train, X_test, y_test)
print(cv_score)

Fit time: 39.781758069992065
Prediction time: 0.2397933006286621

0.9686904761904762


$\textbf{Studying Epochs:}$

$\textbf{Studying Learning Rate:}$

In [ ]:
## Hyper-parameter tunning
param_grid = {
    "model__learning_rate_init": [1e-5, 1e-4, 1e-3, 5e-3, 1e-2, 1e-1, 2e-1, 5e-1],
}
cv_mlp_results, loss_curves, validation_score_curves = cv.hyper_param_tune_one_fold(X_train, y_train, X_test, y_test, param_grid, 
                                                                                    reduced=False, train_size=None, 
                                                                                    return_loss_curves=True)
cv_mlp_results

Params:  {'model__learning_rate_init': 1e-05}
Fit time: 201.12968611717224
Prediction time: 0.2282404899597168



In [ ]:
## Plot Loss curves and validation score curves
for k, v in loss_curves.items():
    fig, ax = plt.subplots(figsize=(12, 3), nrows=1, ncols=2)
    epochs = [i for i in range(1, len(v)+1)]
    ax[0].plot(epochs, loss_curves[k], color='blue')
    ax[1].plot(epochs, validation_score_curves[k], color='blue')
    ax[0].set_xlabel("Epochs")
    ax[0].set_ylabel("Loss in Training")
    ax[0].grid(True)
    ax[1].set_xlabel("Epochs")
    ax[1].set_ylabel("Validation Score")
    ax[1].grid(True)
    fig.suptitle("Params: " + k, fontsize=10)
    plt.show()
    print()

Next Steps:
- Write conclusions about learning rate study.
- Study epochs
- Study hidden layer sizes
- Study activation
- Study alpha
- Study batch size